# 00 — Setup

Run this notebook first to:
1. Install required Python packages.
2. Configure Hugging Face cache paths to point at the `hf/` directory at scratch root.
3. Verify that the GPU and model checkpoints are reachable.

**Launch Jupyter from `notebooks/`** (or open these files directly in VS Code), so the
`../data` and `../results` paths used throughout notebooks 01–04 resolve correctly. The
HF cache paths in Cell 2 locate the project root themselves, so they work either way.

**Hardware / software** (from `environment.json`):

| | |
|---|---|
| python | 3.9.12 |
| torch | 2.7.0+cu126 |
| transformers | 4.52.4 |
| accelerate | 1.10.1 |
| model | EleutherAI/pythia-12b-deduped |
| revision | step143000 |
| dtype | torch.float16 |
| GPU | NVIDIA H200 (150 GB) |

In [ ]:
# Cell 1 — install dependencies
%pip install torch transformers
%pip install -U "huggingface_hub[cli]"
%pip install accelerate
%pip install datasets tqdm pandas numpy matplotlib scipy statsmodels scikit-learn python-docx

Defaulting to user installation because normal site-packages is not writeable
  Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.8.0
    Uninstalling huggingface-hub-1.8.0:
      Successfully uninstalled huggingface-hub-1.8.0
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Using cached huggingface_hub-1.8.0-py3-none-any.whl (625 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.2
    Uninstalling huggingface-hub-0.36.2:
      Successfully uninstalled huggingface-hub-0.36.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.52.4 requires huggingface-hub<1.0,>=0.30.0, but you have huggingface-hub 1.8.0 whi

In [4]:
# Cell 2 — point Hugging Face caches at the local hf/ directory
import os

# Locate the project root (the directory containing hf/hub) by walking up from the
# working directory, so these resolve whether Jupyter was launched from the scratch
# root or from notebooks/. Relative "./hf" silently pointed at notebooks/hf before.
ROOT = os.getcwd()
while ROOT != "/" and not os.path.isdir(os.path.join(ROOT, "hf", "hub")):
    ROOT = os.path.dirname(ROOT)
assert os.path.isdir(os.path.join(ROOT, "hf", "hub")), "could not locate <root>/hf/hub"

# The HF cache is <root>/hf/hub (≈49 GB of Pythia-deduped weights, 70M–12B).
os.environ["HF_HOME"] = os.path.join(ROOT, "hf")
os.environ["HF_HUB_CACHE"] = os.path.join(ROOT, "hf", "hub")
os.environ["HF_DATASETS_CACHE"] = os.path.join(ROOT, "hf", "datasets")

# Make the local bin (where `hf` CLI lives) reachable.
os.environ['PATH'] += ':/storage/home/hcoda1/9/kzhang430/.local/bin'

print("ROOT =", ROOT)
print("HF_HOME =", os.environ["HF_HOME"])
print("HF_HUB_CACHE =", os.environ["HF_HUB_CACHE"])

ROOT = /storage/scratch1/9/kzhang430
HF_HOME = /storage/scratch1/9/kzhang430/hf
HF_HUB_CACHE = /storage/scratch1/9/kzhang430/hf/hub


In [5]:
# Cell 3 — sanity check: do the local model checkpoints exist?
import os
hub = os.environ["HF_HUB_CACHE"]
expected = [
    "models--EleutherAI--pythia-70m-deduped",
    "models--EleutherAI--pythia-160m-deduped",
    "models--EleutherAI--pythia-410m-deduped",
    "models--EleutherAI--pythia-1b-deduped",
    "models--EleutherAI--pythia-1.4b-deduped",
    "models--EleutherAI--pythia-2.8b-deduped",
    "models--EleutherAI--pythia-6.9b-deduped",
    "models--EleutherAI--pythia-12b-deduped",
]
for m in expected:
    p = os.path.join(hub, m)
    ok = os.path.isdir(p)
    print(f"  [{'OK' if ok else 'MISSING'}] {m}")

  [OK] models--EleutherAI--pythia-70m-deduped
  [OK] models--EleutherAI--pythia-160m-deduped
  [OK] models--EleutherAI--pythia-410m-deduped
  [OK] models--EleutherAI--pythia-1b-deduped
  [OK] models--EleutherAI--pythia-1.4b-deduped
  [OK] models--EleutherAI--pythia-2.8b-deduped
  [OK] models--EleutherAI--pythia-6.9b-deduped
  [OK] models--EleutherAI--pythia-12b-deduped


In [6]:
# Cell 4 — confirm HF auth (only needed for gated models; Pythia is open).
!hf auth whoami || echo "(no auth — fine for Pythia, which is public)"

A new version of huggingface_hub (1.27.0) is available! You are using version 1.8.0.
To update, run: pip install -U huggingface_hub

user:  wristycargo


In [7]:
# Cell 5 — check that PyTorch sees the GPU.
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("mem (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

torch: 2.7.0+cu126
cuda available: True
device: NVIDIA H200
mem (GB): 150.111977472


Once all of the above prints OK, you're ready to run:

1. `01_corpus_scan.ipynb` — (re)extract Pile arith matches and integer counts.
2. `02_number_distribution.ipynb` — Fig 1, Fig 4.
3. `03_metrics_powerlaw.ipynb` — Fig 2, Fig 3.
4. `04_arithmetic_main.ipynb` — Fig 5, Fig 6, Sec 3.5/3.6, all Tables, Supp.